In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.find-events.detect-dangerous-animal",
        description="Detect dangerous, large animals in residential area security camera footage",
        worker_release="qwen3-instruct",
        text_prompt="""
          Analyze the provided video or image footage to determine if it depicts a large, potentially dangerous wild animal within a residential environment.

          Read the following definitions carefully before making your decision:
          dangerous_animal: This label is STRICTLY reserved for scenes where a large, potentially dangerous wild animal is physically visible actively encroaching on a residential property or human-populated area. Target animals include, but are not limited to, bears, mountain lions/cougars, wolves, coyotes, moose, or alligators. Residential environments include front porches, driveways, backyards, sidewalks, or immediately adjacent to homes and vehicles.
          safe_environment_or_other: This applies to any scene that does not feature a large, dangerous wild animal in a residential setting. This EXPLICITLY INCLUDES scenes featuring normal domestic pets (dogs, cats), small or harmless wildlife (squirrels, raccoons, birds, rabbits), humans walking by, delivery drivers, empty porches, or animals entirely contained within their natural, unpopulated habitats (deep forests, nature reserves).

          Task:
          Identify which category best describes the footage. Output ONLY one of the following exact labels: dangerous_animal or safe_environment_or_other. Do not include any other text or explanation.
          CRITICAL EXCLUSION: You must differentiate between large domestic animals and wild predators. A large dog walking on a porch does NOT count as a wildlife hazard. Furthermore, the animal must be in a human environment; a bear walking through a deep, unpopulated forest should be labeled safe_environment_or_other.

          CRITICAL FORMATTING: You must not output anything other than exactly "dangerous_animal" or "safe_environment_or_other". Do not include punctuation, markdown formatting, conversational filler, or explanations of any kind.

        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            fps=10,
            image_size=512
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Video

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.find-events.detect-dangerous-animal:latest",
   )
])

video_path = ... # Add path to video

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_video_path = Path(video_path)
   job = endpoint.upload(sample_video_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")